# Lesson 3: Reflection and Blogpost Writing

## Setup

In [1]:
llm_config = {"model": "gpt-3.5-turbo"}

## The task!

In [2]:
task = '''
        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [4]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])

In [5]:
print(reply)

Title: "Unlocking the Power of AI with DeepLearning.AI"

Dive into the future of technology with DeepLearning.AI, a pioneering platform revolutionizing the world of artificial intelligence. Offering cutting-edge courses and resources, DeepLearning.AI empowers learners to delve into the realms of machine learning and neural networks. From beginner to advanced levels, individuals can upskill and stay ahead in the rapidly evolving tech landscape. Join this dynamic community to unleash your potential, whether you're an aspiring data scientist or seasoned professional. Explore the endless possibilities of AI and embark on a transformative learning journey with DeepLearning.AI today.


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: "Unlocking the Power of AI with DeepLearning.AI"

Dive into the future of technology with DeepLearning.AI, a pioneering platform revolutionizing the world of artificial intelligence. Offering cutting-edge courses and resources, DeepLearning.AI empowers learners to delve into the realms of machine learning and neural networks. From beginner to advanced levels, individuals can upskill and stay ahead in the rapidly evolving tech landscape. Join this dynamic community to unleash your potential, whether you're an aspiring data scientist or seasoned professional. Explore the endless possibilities of AI and embark on a transformative learning journey with DeepLearning.AI today.

----------------------------------------------------

## Nested chat

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: "Unlocking the Power of AI with DeepLearning.AI"

Dive into the future of technology with DeepLearning.AI, a pioneering platform revolutionizing the world of artificial intelligence. Offering cutting-edge courses and resources, DeepLearning.AI empowers learners to delve into the realms of machine learning and neural networks. From beginner to advanced levels, individuals can upskill and stay ahead in the rapidly evolving tech landscape. Join this dynamic community to unleash your potential, whether you're an aspiring data scientist or seasoned professional. Explore the endless possibilities of AI and embark on a transformative learning journey with DeepLearning.AI today.

----------------------------------------------------


--------------------------------------------------------------------------------
Meta Reviewer (to Critic):

Aggregated Feedback:
1. SEO Reviewer: Incorporate targeted keywords, craft a compelling meta description, and include internal linking to related content within the website.
2. Legal Reviewer: Ensure compliance with trademark laws, avoid misleading claims about DeepLearning.AI's capabilities, and verify authenticity of endorsements and testimonials.
3. Ethics Reviewer: Avoid exaggerated or misleading claims about benefits, ensure authenticity of testimonials, and provide transparency regarding biases or conflicts of interest.

Final Suggestions:
1. Incorporate targeted keywords and internal linking for SEO purposes, ensuring the content is easily discoverable online.
2. Prioritize legal compliance by verifying the use of "DeepLearning.AI" does not infringe upon trademarks or IP rights, and avoid any misleading claims or false endorsements.
3. Uphold ethical standards by providi

## Get the summary

In [15]:
print(res.summary)

Title: "Empower Your Future: Discovering the World of DeepLearning.AI"

Unlock the potential of AI with DeepLearning.AI, a leading platform in machine learning and neural networks. Dive into our range of courses tailored for beginners and experts looking to excel in the tech industry. Join a vibrant community and explore the limitless possibilities of AI to advance your career. Enhance your skills, whether you're an aspiring data scientist or seasoned professional, and stay at the forefront of technological innovation. Start your transformative learning journey today with DeepLearning.AI and embrace a future powered by AI.
